# Observing Blocks: Spanned parameter space

Owner: **Chris Suberlak** <br>
Last Verified to Run: **2025-12-12** <br>


This notebook allows to plot the parameter space occupied by observing parameters for a chosen science program. The date range and BLOCK name can be updated on the left panel.

In [ ]:
# Times Square Parameters
day_obs_min = 20250828
day_obs_max = 20251201
program_constraint = ''
observation_constraint = ''
target_constraint = ''
def normalize_constraint(value):
    """
    Normalize constraint values from Times Square parameters.
    Converts empty strings, 'None', 'null', etc. to Python None.
    """
    if value is None:
        return None
    if isinstance(value, str):
        stripped = value.strip()
        if stripped == '' or stripped.lower() in ('none', 'null', 'undefined'):
            return None
        return stripped
    return value


# Apply normalization to all constraints at the start of the notebook
program_constraint = normalize_constraint(program_constraint)
observation_constraint = normalize_constraint(observation_constraint)
target_constraint = normalize_constraint(target_constraint)

In [ ]:
from lsst.summit.utils import ConsDbClient
import os
import numpy as np

# Only modify no_proxy if it exists (RSP environment)
if "no_proxy" in os.environ:
    os.environ["no_proxy"] += ",.consdb"

consdb_url = "http://consdb-pq.consdb:8080/consdb"
client = ConsDbClient(consdb_url)

In [ ]:
def print_query_summary(day_obs_min, day_obs_max, target_constraint=None, 
                        observation_constraint=None, program_constraint=None):
    """Print a summary of the query constraints being applied."""
    
    print(f'Currently querying consDB for date range {day_obs_min} - {day_obs_max}')
    
    constraints = []
    if target_constraint is not None and str(target_constraint).strip() != '':
        constraints.append(f"  • target_name: '{target_constraint}'")
    if observation_constraint is not None and str(observation_constraint).strip() != '':
        constraints.append(f"  • observation_reason: '{observation_constraint}'")
    if program_constraint is not None and str(program_constraint).strip() != '':
        constraints.append(f"  • science_program: '{program_constraint}'")
    
    if constraints:
        print("with the following constraints:")
        for c in constraints:
            print(c)
    else:
        print("with no additional constraints.")

print_query_summary(
    day_obs_min, day_obs_max,
    target_constraint=target_constraint,
    observation_constraint=observation_constraint,
    program_constraint=program_constraint
)

If you would like to choose a different science program, search for a desired phrase below, and copy-paste the desired block name (e.g., "BLOCK-T123") into the "program_constraint" field on the left, and click "update". This will recompute the notebook for that science program. The same is supported for target name  and observation reason.

**Wildcard expressions with `*` are supported:**

| Pattern | Meaning | Example |
|---------|---------|---------|
| `value` | exact match | `lowdust` matches only "lowdust" |
| `*value` | ends with | `*dust` matches "lowdust", "highdust" |
| `value*` | starts with | `LMC_*` matches "LMC_field1", "LMC_test" |
| `*value*` | contains | `*dust*` matches "lowdust_v1", "my_dust_field" |
| `prefix_*suffix` | starts with prefix, ends with suffix | `LMC_*lowdust` matches "LMC_field_lowdust" |
| `prefix_*mid*` | starts with prefix, contains mid | `LMC_*dust*` matches "LMC_dust_v1", "LMC_lowdust_test" |


In [ ]:
from IPython.display import HTML, display
import json
import numpy as np
import random
import re

def build_pattern_constraint(pattern, column_name, case_insensitive=True):
    """
    Convert user-friendly wildcard patterns to SQL constraint for any column.
    
    Patterns:
        'value'         -> exact match
        '*value'        -> ends with 'value'
        'value*'        -> starts with 'value'
        '*value*'       -> contains 'value'
        'prefix_*value' -> starts with 'prefix_' and ends with 'value'
        'prefix_*mid*'  -> starts with 'prefix_', contains 'mid'
        
    The '*' is converted to '.*' for regex matching.
    
    Parameters:
        pattern: str or None - the user's search pattern
        column_name: str - the SQL column name (e.g., 't.target_name', 't.observation_reason')
        case_insensitive: bool - whether to ignore case
        
    Returns:
        str - SQL constraint string (including leading ' AND ') or empty string
    """
    # Handle None, empty strings, and 'None' string
    if pattern is None:
        return ""
    
    stripped = str(pattern).strip()
    
    if stripped == '' or stripped.lower() in ('none', 'null', 'undefined'):
        return ""
    
    # Check if it's a simple pattern (no wildcards)
    if '*' not in stripped:
        if case_insensitive:
            return f" AND LOWER({column_name}) = LOWER('{stripped}')"
        else:
            return f" AND {column_name} = '{stripped}'"
    
    # Convert wildcard pattern to regex:
    # 1. Escape any special regex characters (except our wildcards)
    # 2. Replace '*' with '.*'
    
    special_chars = r'\.^$+?{}[]|()'
    
    regex_pattern = ''
    for char in stripped:
        if char == '*':
            regex_pattern += '.*'
        elif char in special_chars:
            regex_pattern += '\\' + char
        else:
            regex_pattern += char
    
    # Anchor the pattern if it doesn't start/end with wildcard
    if not stripped.startswith('*'):
        regex_pattern = '^' + regex_pattern
    if not stripped.endswith('*'):
        regex_pattern = regex_pattern + '$'
    
    # Use ~* for case-insensitive, ~ for case-sensitive
    regex_op = '~*' if case_insensitive else '~'
    
    return f" AND {column_name} {regex_op} '{regex_pattern}'"


def build_query_constraints(program_constraint=None, 
                            observation_constraint=None,
                            target_constraint=None, 
                            case_insensitive=False):
    """
    Build SQL constraints for both target_name and observation_reason.
    
    Parameters:
        program_constraint: str or None - pattern for t.science_program
        observation_constraint: str or None - pattern for t.observation_reason
        target_constraint: str or None - pattern for t.target_name
        case_insensitive: bool - whether to ignore case
        
    Returns:
        str - combined SQL constraint string
    """
    constraints = ""
    constraints += build_pattern_constraint(program_constraint, 't.science_program', case_insensitive)
    constraints += build_pattern_constraint(observation_constraint, 't.observation_reason', case_insensitive)
    constraints += build_pattern_constraint(target_constraint, 't.target_name', case_insensitive)

    return constraints

# Generate unique ID to avoid conflicts
unique_id = random.randint(10000, 99999)


query = f'''
SELECT 
  t.day_obs, 
  t.science_program,
  t.observation_reason,
  t.target_name
FROM 
  cdb_lsstcam.visit1 AS t
WHERE  
  t.day_obs > {day_obs_min} AND
  t.day_obs < {day_obs_max}
'''

query += build_query_constraints(program_constraint=program_constraint,
                                 observation_constraint=observation_constraint,
                                 target_constraint=target_constraint
                                )

df = client.query(query)

# Check if query returned any results
if len(df) == 0:
    print("⚠️ No data found for the specified constraints:")
    print_query_summary(day_obs_min, day_obs_max, 
                        target_constraint=target_constraint,
                        observation_constraint=observation_constraint,
                        program_constraint=program_constraint)
    no_data = True
else:
    no_data = False

if not no_data:
    unique_programs = np.unique(df['science_program'])

    # Build structured data for JavaScript (now including targets)
    programs_data = []
    for program in unique_programs:
        mask = df['science_program'] == program
        program_rows = df[mask]
        unique_reasons = np.unique(program_rows['observation_reason'])

        # Build reasons with their targets
        reasons_data = []
        for reason in unique_reasons:
            reason_mask = program_rows['observation_reason'] == reason
            targets = np.unique(program_rows[reason_mask]['target_name'])
            # Filter out None, empty strings, and 'None' strings
            valid_targets = [str(t) for t in targets if t is not None and str(t).strip() != '' and str(t).lower() != 'none']
            reasons_data.append({
                'reason': str(reason),
                'targets': valid_targets
            })

        programs_data.append({
            'program': str(program),
            'reasons': reasons_data,
            'count': int(len(program_rows))
        })

    # Create HTML with inline JavaScript
    html_inline = f"""
    <div style="max-width: 1200px; margin: 20px auto;">
        <div style="background: linear-gradient(135deg, #667eea 0%, #764ba2 100%); 
                    color: white; padding: 20px; border-radius: 8px;">
            <h2 style="margin: 0 0 8px 0;">📋 Available Science Programs</h2>
            <p style="margin: 0; opacity: 0.95;">Date range: {day_obs_min} to {day_obs_max}</p>
            <button id="toggle-btn-{unique_id}" 
                    style="margin-top: 15px; padding: 10px 20px; background: white; color: #667eea; 
                           border: none; border-radius: 4px; font-size: 14px; font-weight: bold; 
                           cursor: pointer; box-shadow: 0 2px 4px rgba(0,0,0,0.2);
                           transition: all 0.3s ease;">
                Show Available Programs
            </button>
        </div>

        <div id="programs-container-{unique_id}" style="display: none;">
            <div style="background: white; padding: 15px; border-bottom: 1px solid #dee2e6;">
                <input type="text" id="search-input-{unique_id}" 
                       placeholder="Search programs, observation reasons, or targets..." 
                       style="width: 100%; padding: 10px; border: 2px solid #ced4da; 
                              border-radius: 4px; font-size: 14px; box-sizing: border-box;">
                <div id="status-{unique_id}" style="margin-top: 8px; padding: 8px; border-radius: 4px; 
                                               background: #d1ecf1; color: #0c5460; font-size: 13px;">
                    💡 Start typing to filter. Click on observation reasons to expand targets.
                </div>
            </div>

            <div id="results-{unique_id}" style="background: white; padding: 20px; max-height: 600px; 
                                            overflow-y: auto; border-radius: 0 0 8px 8px; box-shadow: 0 2px 8px rgba(0,0,0,0.1);">
            </div>
        </div>
    </div>

    <script>
    (function() {{
        const programsData = {json.dumps(programs_data)};
        const container = document.getElementById('programs-container-{unique_id}');
        const toggleBtn = document.getElementById('toggle-btn-{unique_id}');
        const resultsContainer = document.getElementById('results-{unique_id}');
        const searchInput = document.getElementById('search-input-{unique_id}');
        const statusDiv = document.getElementById('status-{unique_id}');

        let initialized = false;

        // Toggle visibility
        toggleBtn.addEventListener('click', function() {{
            if (container.style.display === 'none' || container.style.display === '') {{
                container.style.display = 'block';
                toggleBtn.textContent = 'Hide Available Programs';
                toggleBtn.style.background = '#f8f9fa';

                if (!initialized) {{
                    initializePrograms();
                    initialized = true;
                }}
            }} else {{
                container.style.display = 'none';
                toggleBtn.textContent = 'Show Available Programs';
                toggleBtn.style.background = 'white';
            }}
        }});

        function escapeHtml(text) {{
            const div = document.createElement('div');
            div.textContent = text;
            return div.innerHTML;
        }}

        function initializePrograms() {{
            programsData.forEach(function(prog, progIdx) {{
                const div = document.createElement('div');
                div.className = 'prog-block-{unique_id}';
                div.dataset.name = prog.program.toLowerCase();

                // Build reasons HTML
                let reasonsHtml = '';
                prog.reasons.forEach(function(r, reasonIdx) {{
                    // Build targets HTML
                    let targetsHtml = '';
                    const hasTargets = r.targets.length > 0;

                    if (hasTargets) {{
                        r.targets.forEach(function(t) {{
                            targetsHtml += '<div class="target-item-{unique_id}" data-target="' + escapeHtml(t.toLowerCase()) + '" ' +
                                   'style="padding: 4px 8px; margin: 2px 0 2px 15px; background: #fff3cd; ' +
                                   'border-left: 2px solid #ffc107; border-radius: 3px; font-size: 12px; color: #856404;">' +
                                   escapeHtml(t) + '</div>';
                        }});
                    }} else {{
                        targetsHtml = '<div style="padding: 4px 8px; margin: 2px 0 2px 15px; ' +
                                      'font-size: 12px; color: #6c757d; font-style: italic;">' +
                                      'No target names specified</div>';
                    }}

                    const targetCountText = hasTargets ? 
                        '(' + r.targets.length + ' target' + (r.targets.length > 1 ? 's' : '') + ')' : 
                        '(no target names specified)';

                    reasonsHtml += '<div class="reason-item-{unique_id}" data-reason="' + escapeHtml(r.reason.toLowerCase()) + '" ' +
                           'data-targets="' + escapeHtml(r.targets.join(',').toLowerCase()) + '">' +
                           '<div class="reason-header-{unique_id}" ' +
                           'style="padding: 6px 10px; margin: 3px 0; background: #f8f9fa; ' +
                           'border-left: 3px solid #667eea; border-radius: 3px; font-size: 13px; cursor: pointer;">' +
                           '<span class="arrow-{unique_id}" style="margin-right: 8px;">▶</span>' + escapeHtml(r.reason) + 
                           '<span style="color: #6c757d; margin-left: 8px; font-size: 11px;">' + targetCountText + '</span>' +
                           '</div>' +
                           '<div class="targets-container-{unique_id}" style="display: none;">' + 
                           targetsHtml + '</div></div>';
                }});

                div.innerHTML = 
                    '<div style="border: 1px solid #dee2e6; border-radius: 6px; margin-bottom: 15px; overflow: hidden;">' +
                        '<div style="background: #f8f9fa; padding: 12px 15px; border-bottom: 1px solid #dee2e6;">' +
                            '<strong style="font-size: 16px; color: #495057;">' + escapeHtml(prog.program) + '</strong>' +
                            '<span style="color: #6c757d; margin-left: 10px; font-size: 13px;">' + prog.count + ' visits</span>' +
                        '</div>' +
                        '<div style="padding: 10px 15px;">' + reasonsHtml + '</div>' +
                    '</div>';

                resultsContainer.appendChild(div);
            }});

            // Add click handlers for expanding/collapsing targets
            const reasonHeaders = resultsContainer.querySelectorAll('.reason-header-{unique_id}');
            reasonHeaders.forEach(function(header) {{
                header.addEventListener('click', function() {{
                    const targetsDiv = this.nextElementSibling;
                    const arrow = this.querySelector('.arrow-{unique_id}');
                    if (targetsDiv.style.display === 'none' || targetsDiv.style.display === '') {{
                        targetsDiv.style.display = 'block';
                        arrow.textContent = '▼';
                    }} else {{
                        targetsDiv.style.display = 'none';
                        arrow.textContent = '▶';
                    }}
                }});
            }});
        }}

        searchInput.addEventListener('input', function() {{
            const term = searchInput.value.toLowerCase().trim();
            const blocks = resultsContainer.querySelectorAll('.prog-block-{unique_id}');

            if (!term) {{
                blocks.forEach(function(b) {{
                    b.style.display = 'block';
                    const reasons = b.querySelectorAll('.reason-item-{unique_id}');
                    reasons.forEach(function(r) {{
                        r.style.display = 'block';
                        r.querySelector('.reason-header-{unique_id}').style.background = '#f8f9fa';
                        const targets = r.querySelectorAll('.target-item-{unique_id}');
                        targets.forEach(function(t) {{
                            t.style.background = '#fff3cd';
                        }});
                    }});
                }});
                statusDiv.textContent = '💡 Start typing to filter. Click on observation reasons to expand targets.';
                statusDiv.style.background = '#d1ecf1';
                statusDiv.style.color = '#0c5460';
                return;
            }}

            let matched = 0;
            blocks.forEach(function(block) {{
                const name = block.dataset.name;
                const nameMatch = name.includes(term);
                const reasons = block.querySelectorAll('.reason-item-{unique_id}');
                let hasMatch = nameMatch;

                reasons.forEach(function(r) {{
                    const reasonMatch = r.dataset.reason.includes(term);
                    const targetsStr = r.dataset.targets || '';
                    const targetMatch = targetsStr.includes(term);
                    const reasonHeader = r.querySelector('.reason-header-{unique_id}');
                    const targetsContainer = r.querySelector('.targets-container-{unique_id}');
                    const targetItems = r.querySelectorAll('.target-item-{unique_id}');

                    if (reasonMatch || targetMatch) {{
                        r.style.display = 'block';
                        reasonHeader.style.background = '#e3f2fd';
                        hasMatch = true;

                        // If target matched, expand and highlight
                        if (targetMatch) {{
                            targetsContainer.style.display = 'block';
                            reasonHeader.querySelector('.arrow-{unique_id}').textContent = '▼';
                            targetItems.forEach(function(t) {{
                                if (t.dataset.target.includes(term)) {{
                                    t.style.background = '#c8e6c9';
                                }} else {{
                                    t.style.background = '#fff3cd';
                                }}
                            }});
                        }}
                    }} else if (nameMatch) {{
                        r.style.display = 'block';
                        reasonHeader.style.background = '#f8f9fa';
                    }} else {{
                        r.style.display = 'none';
                    }}
                }});

                block.style.display = hasMatch ? 'block' : 'none';
                if (hasMatch) matched++;
            }});

            statusDiv.textContent = matched > 0 ? '✓ Found ' + matched + ' program(s)' : '⚠️ No matches';
            statusDiv.style.background = matched > 0 ? '#d4edda' : '#fff3cd';
            statusDiv.style.color = matched > 0 ? '#155724' : '#856404';
        }});
    }})();
    </script>
    """

    display(HTML(html_inline))

    print(f"✓ Ready to display {len(programs_data)} science programs")
    print(f"Date range: {day_obs_min} to {day_obs_max}")
    print("Click 'Show Available Programs' to view the list")

In [ ]:
from bokeh.plotting import figure, output_file, save, show, output_notebook
from bokeh.layouts import gridplot, column, row
from bokeh.models import (HoverTool, ColumnDataSource, Div, Slider, CustomJS, 
                          TextInput, Range1d, CheckboxGroup, Button, RangeSlider)
import numpy as np
from datetime import datetime
from lsst.summit.utils import ConsDbClient
import matplotlib.pyplot as plt 
from matplotlib.gridspec import GridSpec

consdb_url = "http://consdb-pq.consdb:8080/consdb"
client = ConsDbClient(consdb_url)

query = f'''
SELECT 
  t.visit_id, t.day_obs, 
  t.seq_num, t.physical_filter,
  t.azimuth, t.altitude, t.airmass, t.science_program,
  t.observation_reason, t.dimm_seeing,
  q.physical_rotator_angle,
  q.aos_fwhm, q.psf_sigma_median, q.donut_blur_fwhm,
  q.ringss_seeing
FROM 
  cdb_lsstcam.visit1 AS t,
  cdb_lsstcam.visit1_quicklook AS q
WHERE 
  t.visit_id = q.visit_id 
  AND t.day_obs > {day_obs_min} 
  AND t.day_obs < {day_obs_max}
'''
query += build_query_constraints(program_constraint=program_constraint,
                                 observation_constraint=observation_constraint,
                                 target_constraint=target_constraint
                                )

df = client.query(query)

# Check if query returned any results
if len(df) == 0:
    print("⚠️ No data found for the specified constraints.")
    no_data = True
else:
    no_data = False

if not no_data:

    max_day_obs = max(np.unique(df['day_obs']))

    # Enable notebook output
    output_notebook()

    # Band colors
    band_colors = {
        "u": "#0c71ff",
        "g": "#49be61",
        "r": "#c61c00",
        "i": "#ffc200",
        "z": "#f341a2",
        "y": "#5d0000",
    }

    # Variables to plot
    variables = ['altitude', 'airmass', 
                 'dimm_seeing', 'physical_rotator_angle',
                 'aos_fwhm', 'psf_fwhm_asec', 'donut_blur_fwhm']

    labels = ['Altitude', 'Airmass', 'DIMM Seeing', 'Rotator',
              'AOS FWHM', 'PSF FWHM', 'Donut Blur']

    # Calculate PSF FWHM from psf_sigma_median
    SIGMA2FWHM = np.sqrt(8 * np.log(2))
    pixToArcseconds = 0.199225
    df['psf_fwhm_asec'] = df["psf_sigma_median"].astype(float) * SIGMA2FWHM * pixToArcseconds

    # Create mask for rows with valid data in all variables
    mask = np.ones(len(df), dtype=bool)
    for var in variables:
        col_data = df[var]
        if hasattr(col_data, 'mask'):
            mask &= ~col_data.mask
        mask &= ~np.isnan(col_data.astype(float, copy=False))

    df_clean = df[mask]
    df_clean_pd = df_clean.to_pandas()

    # Extract bands and colors
    bands = [pf.split('_')[0] for pf in df_clean['physical_filter']]
    colors = [band_colors[band] for band in bands]

    n_vars = len(variables)

    # Prepare data
    df_pd = df_clean_pd.copy()
    df_pd['band'] = bands
    df_pd['color'] = colors
    df_pd['day_obs'] = df_clean['day_obs'].astype(int)
    df_pd['science_program'] = df_clean['science_program']

    # Get available dates
    available_dates = sorted(df_pd['day_obs'].unique())
    n_dates = len(available_dates)

    print(f"Available dates in dataset: {n_dates} unique dates")
    print(f"Date range: {available_dates[0]} to {available_dates[-1]}")

    # Find indices for initial day_obs values
    try:
        start_idx = available_dates.index(day_obs_min) if day_obs_min in available_dates else 0
    except ValueError:
        start_idx = 0

    try:
        end_idx = available_dates.index(day_obs_max) if day_obs_max in available_dates else n_dates - 1
    except ValueError:
        end_idx = n_dates - 1

    # Build program/reasons data for checkboxes
    unique_programs = list(np.unique(df['science_program']))
    program_reasons = {}
    for prog in unique_programs:
        mask_prog = df['science_program'] == prog
        reasons = np.unique(df[mask_prog]['observation_reason'])
        program_reasons[prog] = list(reasons)

    checkbox_labels = []
    for prog in unique_programs:
        reasons_str = ", ".join(str(r) for r in program_reasons[prog])
        checkbox_labels.append(f"{prog}: {reasons_str}")

    # Create checkbox group
    program_checkbox = CheckboxGroup(
        labels=checkbox_labels,
        active=list(range(len(unique_programs))),
        width=600
    )

    # Create initial filtered data
    initial_mask = (df_pd['day_obs'] >= available_dates[start_idx]) & (df_pd['day_obs'] <= available_dates[end_idx])
    df_filtered = df_pd[initial_mask].copy()

    source = ColumnDataSource(df_filtered)
    source_full = ColumnDataSource(df_pd)
    available_dates_source = ColumnDataSource({'dates': available_dates})

    # Create grid of plots
    plots = []
    plot_width = 250
    plot_height = 250

    # Calculate variable bounds
    var_bounds = {}
    for var in variables:
        var_min = float(df_pd[var].min())
        var_max = float(df_pd[var].max())
        padding = (var_max - var_min) * 0.05
        var_bounds[var] = (var_min - padding, var_max + padding)

    # Pre-create shared Range1d objects
    shared_ranges = {}
    for idx, var in enumerate(variables):
        shared_ranges[idx] = Range1d(start=var_bounds[var][0], end=var_bounds[var][1])

    # Store histogram sources
    hist_sources = {}

    for i in range(n_vars):
        row_plots = []
        for j in range(n_vars):
            if j > i:
                row_plots.append(None)
                continue

            if i == j:
                n_bins = 100
                hist, edges = np.histogram(df_filtered[variables[i]], bins=n_bins)

                hist_data = {
                    'top': hist,
                    'left': edges[:-1],
                    'right': edges[1:],
                    'bottom': [0] * len(hist)
                }
                hist_source = ColumnDataSource(hist_data)
                hist_sources[i] = hist_source

                p = figure(width=plot_width, height=plot_height,
                          toolbar_location='above',
                          tools='pan,wheel_zoom,box_zoom,reset,save',
                          x_range=shared_ranges[i])

                p.quad(top='top', bottom='bottom', left='left', right='right',
                      source=hist_source,
                      fill_color='skyblue', line_color='black', alpha=0.7)

                p.yaxis.axis_label = 'Count'
                p.yaxis.axis_label_standoff = 10

                if i == n_vars - 1:
                    p.xaxis.axis_label = labels[j]
                else:
                    p.xaxis.major_label_text_font_size = '0pt'

            else:
                p = figure(width=plot_width, height=plot_height,
                          toolbar_location='above',
                          tools='pan,wheel_zoom,box_zoom,reset,save,hover',
                          x_range=shared_ranges[j],
                          y_range=shared_ranges[i])

                p.scatter(variables[j], variables[i], source=source,
                         size=5, color='color', alpha=0.6, marker='circle')

                hover = p.select_one(HoverTool)
                hover.tooltips = [
                    ('Band', '@band'),
                    ('day_obs', '@day_obs'),
                    ('Program', '@science_program'),
                    (labels[j], f'@{{{variables[j]}}}{{0.000}}'),
                    (labels[i], f'@{{{variables[i]}}}{{0.000}}'),
                ]

                if i == n_vars - 1:
                    p.xaxis.axis_label = labels[j]
                else:
                    p.xaxis.major_label_text_font_size = '0pt'

                if j == 0:
                    p.yaxis.axis_label = labels[i]
                else:
                    p.yaxis.major_label_text_font_size = '0pt'

            row_plots.append(p)

        plots.append(row_plots)

    # Convert hist_sources to a list format for JavaScript
    hist_sources_list = [hist_sources.get(i) for i in range(n_vars)]

    # Helper function to compute initial percentiles
    def compute_percentile_text(data, var_name):
        var_data = data[var_name]
        if len(var_data) == 0:
            return "<span style='font-size:13px; font-family:monospace; color:#999;'>No data</span>"
        p5 = np.nanpercentile(var_data, 5)
        p50 = np.nanpercentile(var_data, 50)
        p95 = np.nanpercentile(var_data, 95)
        return f"<span style='font-size:13px; font-family:monospace;'>{p5:.3f}      {p50:.3f}      {p95:.3f}</span>"

    # Create stats Divs for each variable
    stats_divs = {}
    for idx, var in enumerate(variables):
        initial_stats = compute_percentile_text(df_filtered, var)
        stats_divs[idx] = Div(text=initial_stats, width=250, height=20)

    # Convert to list for JavaScript
    stats_divs_list = [stats_divs[i] for i in range(n_vars)]

    # JavaScript function to compute percentiles
    percentile_js_code = """
    function computePercentile(arr, p) {
        if (arr.length === 0) return NaN;
        const sorted = arr.slice().sort((a, b) => a - b);
        const idx = (p / 100) * (sorted.length - 1);
        const lower = Math.floor(idx);
        const upper = Math.ceil(idx);
        if (lower === upper) return sorted[lower];
        return sorted[lower] + (idx - lower) * (sorted[upper] - sorted[lower]);
    }

    function updateStats(stats_divs, variables, new_data) {
        for (let var_idx = 0; var_idx < variables.length; var_idx++) {
            const stats_div = stats_divs[var_idx];
            if (stats_div) {
                const var_name = variables[var_idx];
                const var_data = new_data[var_name];

                if (!var_data || var_data.length === 0) {
                    stats_div.text = "<span style='font-size:13px; font-family:monospace; color:#999;'>No data</span>";
                    continue;
                }

                const valid_data = var_data.filter(v => !isNaN(v));
                if (valid_data.length === 0) {
                    stats_div.text = "<span style='font-size:13px; font-family:monospace; color:#999;'>No valid data</span>";
                    continue;
                }

                const p5 = computePercentile(valid_data, 5);
                const p50 = computePercentile(valid_data, 50);
                const p95 = computePercentile(valid_data, 95);

                stats_div.text = "<span style='font-size:13px; font-family:monospace;'>" + 
                                p5.toFixed(3) + "      " + p50.toFixed(3) + "      " + p95.toFixed(3) + "</span>";
            }
        }
    }
    """

    # Create date sliders
    date_slider_min = Slider(
        start=0,
        end=n_dates - 1,
        value=start_idx,
        step=1,
        title="Start Date Index",
        width=300
    )

    date_slider_max = Slider(
        start=0,
        end=n_dates - 1,
        value=end_idx,
        step=1,
        title="End Date Index",
        width=300
    )

    date_display_min = Div(text=f"<strong>{available_dates[start_idx]}</strong>", width=120, height=30)
    date_display_max = Div(text=f"<strong>{available_dates[end_idx]}</strong>", width=120, height=30)

    date_text_min = TextInput(value=str(available_dates[start_idx]), title="Or enter date:", width=120)
    date_text_max = TextInput(value=str(available_dates[end_idx]), title="Or enter date:", width=120)

    date_status = Div(
        text=f"<p style='margin: 5px 0; color: #666;'>Showing {len(df_filtered)} of {len(df_pd)} points | {end_idx - start_idx + 1} of {n_dates} dates</p>", 
        width=700, 
        height=30
    )

    # Slider callback
    slider_callback = CustomJS(
        args=dict(
            source=source,
            source_full=source_full,
            slider_min=date_slider_min,
            slider_max=date_slider_max,
            display_min=date_display_min,
            display_max=date_display_max,
            text_min=date_text_min,
            text_max=date_text_max,
            date_status=date_status,
            hist_sources=hist_sources_list,
            variables=variables,
            available_dates=available_dates_source,
            checkbox=program_checkbox,
            programs=unique_programs,
            stats_divs=stats_divs_list
        ),
        code=percentile_js_code + """
        const idx_min = Math.floor(slider_min.value);
        const idx_max = Math.floor(slider_max.value);

        if (idx_min > idx_max) {
            return;
        }

        const dates = available_dates.data['dates'];
        const min_date = dates[idx_min];
        const max_date = dates[idx_max];

        display_min.text = '<strong>' + min_date + '</strong>';
        display_max.text = '<strong>' + max_date + '</strong>';
        text_min.value = min_date.toString();
        text_max.value = max_date.toString();

        const active_indices = checkbox.active;
        const selected_programs = active_indices.map(i => programs[i]);

        const full_data = source_full.data;
        const day_obs = full_data['day_obs'];
        const science_program = full_data['science_program'];
        const filtered_indices = [];

        for (let i = 0; i < day_obs.length; i++) {
            const in_date_range = day_obs[i] >= min_date && day_obs[i] <= max_date;
            const in_program = selected_programs.includes(science_program[i]);

            if (in_date_range && in_program) {
                filtered_indices.push(i);
            }
        }

        const new_data = {};
        for (const key in full_data) {
            new_data[key] = filtered_indices.map(i => full_data[key][i]);
        }
        source.data = new_data;
        source.change.emit();

        // Update statistics
        updateStats(stats_divs, variables, new_data);

        const n_dates_in_range = idx_max - idx_min + 1;
        date_status.text = '<p style="margin: 5px 0; color: #666;">Showing ' + filtered_indices.length + 
                          ' of ' + day_obs.length + ' points | ' +
                          selected_programs.length + ' of ' + programs.length + ' programs | ' +
                          n_dates_in_range + ' of ' + dates.length + ' dates</p>';

        for (let var_idx = 0; var_idx < variables.length; var_idx++) {
            const hist_source = hist_sources[var_idx];
            if (hist_source) {
                const var_name = variables[var_idx];
                const var_data = new_data[var_name];

                if (!var_data || var_data.length === 0) {
                    hist_source.data = {
                        'top': [],
                        'left': [],
                        'right': [],
                        'bottom': []
                    };
                    hist_source.change.emit();
                    continue;
                }

                const var_min = Math.min(...var_data);
                const var_max = Math.max(...var_data);

                const n_bins = 20;
                const bin_width = (var_max - var_min) / n_bins || 1;
                const bins = new Array(n_bins).fill(0);
                const edges_left = [];
                const edges_right = [];

                for (let i = 0; i < n_bins; i++) {
                    edges_left.push(var_min + i * bin_width);
                    edges_right.push(var_min + (i + 1) * bin_width);
                }

                for (let i = 0; i < var_data.length; i++) {
                    const val = var_data[i];
                    if (!isNaN(val)) {
                        const bin_idx = Math.min(Math.floor((val - var_min) / bin_width), n_bins - 1);
                        if (bin_idx >= 0 && bin_idx < n_bins) {
                            bins[bin_idx]++;
                        }
                    }
                }

                hist_source.data = {
                    'top': bins,
                    'left': edges_left,
                    'right': edges_right,
                    'bottom': new Array(n_bins).fill(0)
                };
                hist_source.change.emit();
            }
        }
    """
    )

    # Text callback
    text_callback = CustomJS(
        args=dict(
            source=source,
            source_full=source_full,
            slider_min=date_slider_min,
            slider_max=date_slider_max,
            display_min=date_display_min,
            display_max=date_display_max,
            text_min=date_text_min,
            text_max=date_text_max,
            date_status=date_status,
            hist_sources=hist_sources_list,
            variables=variables,
            available_dates=available_dates_source,
            checkbox=program_checkbox,
            programs=unique_programs,
            stats_divs=stats_divs_list
        ),
        code=percentile_js_code + """
        const target_min = parseInt(text_min.value);
        const target_max = parseInt(text_max.value);

        if (isNaN(target_min) || isNaN(target_max)) {
            date_status.text = '<p style="margin: 5px 0; color: #dc3545;">⚠️ Invalid date format</p>';
            return;
        }

        const dates = available_dates.data['dates'];

        let idx_min = 0;
        let idx_max = dates.length - 1;

        for (let i = 0; i < dates.length; i++) {
            if (dates[i] >= target_min) {
                idx_min = i;
                break;
            }
        }

        for (let i = dates.length - 1; i >= 0; i--) {
            if (dates[i] <= target_max) {
                idx_max = i;
                break;
            }
        }

        if (idx_min > idx_max) {
            date_status.text = '<p style="margin: 5px 0; color: #dc3545;">⚠️ No data in this range</p>';
            return;
        }

        slider_min.value = idx_min;
        slider_max.value = idx_max;

        const min_date = dates[idx_min];
        const max_date = dates[idx_max];

        display_min.text = '<strong>' + min_date + '</strong>';
        display_max.text = '<strong>' + max_date + '</strong>';
        text_min.value = min_date.toString();
        text_max.value = max_date.toString();

        const active_indices = checkbox.active;
        const selected_programs = active_indices.map(i => programs[i]);

        const full_data = source_full.data;
        const day_obs = full_data['day_obs'];
        const science_program = full_data['science_program'];
        const filtered_indices = [];

        for (let i = 0; i < day_obs.length; i++) {
            const in_date_range = day_obs[i] >= min_date && day_obs[i] <= max_date;
            const in_program = selected_programs.includes(science_program[i]);

            if (in_date_range && in_program) {
                filtered_indices.push(i);
            }
        }

        const new_data = {};
        for (const key in full_data) {
            new_data[key] = filtered_indices.map(i => full_data[key][i]);
        }
        source.data = new_data;
        source.change.emit();

        // Update statistics
        updateStats(stats_divs, variables, new_data);

        const n_dates_in_range = idx_max - idx_min + 1;
        date_status.text = '<p style="margin: 5px 0; color: #28a745;">✓ Showing ' + filtered_indices.length + 
                          ' of ' + day_obs.length + ' points | ' +
                          selected_programs.length + ' of ' + programs.length + ' programs | ' +
                          n_dates_in_range + ' of ' + dates.length + ' dates</p>';

        for (let var_idx = 0; var_idx < variables.length; var_idx++) {
            const hist_source = hist_sources[var_idx];
            if (hist_source) {
                const var_name = variables[var_idx];
                const var_data = new_data[var_name];

                if (!var_data || var_data.length === 0) {
                    hist_source.data = {
                        'top': [],
                        'left': [],
                        'right': [],
                        'bottom': []
                    };
                    hist_source.change.emit();
                    continue;
                }

                const var_min = Math.min(...var_data);
                const var_max = Math.max(...var_data);

                const n_bins = 20;
                const bin_width = (var_max - var_min) / n_bins || 1;
                const bins = new Array(n_bins).fill(0);
                const edges_left = [];
                const edges_right = [];

                for (let i = 0; i < n_bins; i++) {
                    edges_left.push(var_min + i * bin_width);
                    edges_right.push(var_min + (i + 1) * bin_width);
                }

                for (let i = 0; i < var_data.length; i++) {
                    const val = var_data[i];
                    if (!isNaN(val)) {
                        const bin_idx = Math.min(Math.floor((val - var_min) / bin_width), n_bins - 1);
                        if (bin_idx >= 0 && bin_idx < n_bins) {
                            bins[bin_idx]++;
                        }
                    }
                }

                hist_source.data = {
                    'top': bins,
                    'left': edges_left,
                    'right': edges_right,
                    'bottom': new Array(n_bins).fill(0)
                };
                hist_source.change.emit();
            }
        }
    """
    )

    # Program filter callback
    program_filter_callback = CustomJS(
        args=dict(
            source=source,
            source_full=source_full,
            checkbox=program_checkbox,
            programs=unique_programs,
            slider_min=date_slider_min,
            slider_max=date_slider_max,
            available_dates=available_dates_source,
            date_status=date_status,
            hist_sources=hist_sources_list,
            variables=variables,
            stats_divs=stats_divs_list
        ),
        code=percentile_js_code + """
        const active_indices = checkbox.active;
        const selected_programs = active_indices.map(i => programs[i]);

        const idx_min = Math.floor(slider_min.value);
        const idx_max = Math.floor(slider_max.value);
        const dates = available_dates.data['dates'];
        const min_date = dates[idx_min];
        const max_date = dates[idx_max];

        const full_data = source_full.data;
        const day_obs = full_data['day_obs'];
        const science_program = full_data['science_program'];
        const filtered_indices = [];

        for (let i = 0; i < day_obs.length; i++) {
            const in_date_range = day_obs[i] >= min_date && day_obs[i] <= max_date;
            const in_program = selected_programs.includes(science_program[i]);

            if (in_date_range && in_program) {
                filtered_indices.push(i);
            }
        }

        const new_data = {};
        for (const key in full_data) {
            new_data[key] = filtered_indices.map(i => full_data[key][i]);
        }

        source.data = new_data;
        source.change.emit();

        // Update statistics
        updateStats(stats_divs, variables, new_data);

        const n_dates_in_range = idx_max - idx_min + 1;
        date_status.text = '<p style="margin: 5px 0; color: #666;">Showing ' + filtered_indices.length + 
                          ' of ' + day_obs.length + ' points | ' + 
                          selected_programs.length + ' of ' + programs.length + ' programs | ' +
                          n_dates_in_range + ' of ' + dates.length + ' dates</p>';

        for (let var_idx = 0; var_idx < variables.length; var_idx++) {
            const hist_source = hist_sources[var_idx];
            if (hist_source) {
                const var_name = variables[var_idx];
                const var_data = new_data[var_name];

                if (!var_data || var_data.length === 0) {
                    hist_source.data = {
                        'top': [],
                        'left': [],
                        'right': [],
                        'bottom': []
                    };
                    hist_source.change.emit();
                    continue;
                }

                const var_min = Math.min(...var_data);
                const var_max = Math.max(...var_data);

                const n_bins = 20;
                const bin_width = (var_max - var_min) / n_bins || 1;
                const bins = new Array(n_bins).fill(0);
                const edges_left = [];
                const edges_right = [];

                for (let i = 0; i < n_bins; i++) {
                    edges_left.push(var_min + i * bin_width);
                    edges_right.push(var_min + (i + 1) * bin_width);
                }

                for (let i = 0; i < var_data.length; i++) {
                    const val = var_data[i];
                    if (!isNaN(val)) {
                        const bin_idx = Math.min(Math.floor((val - var_min) / bin_width), n_bins - 1);
                        if (bin_idx >= 0 && bin_idx < n_bins) {
                            bins[bin_idx]++;
                        }
                    }
                }

                hist_source.data = {
                    'top': bins,
                    'left': edges_left,
                    'right': edges_right,
                    'bottom': new Array(n_bins).fill(0)
                };
                hist_source.change.emit();
            }
        }
    """
    )

    # Select all callback
    select_all_callback = CustomJS(
        args=dict(
            checkbox=program_checkbox, 
            n_programs=len(unique_programs),
            source=source,
            source_full=source_full,
            programs=unique_programs,
            slider_min=date_slider_min,
            slider_max=date_slider_max,
            available_dates=available_dates_source,
            date_status=date_status,
            hist_sources=hist_sources_list,
            variables=variables,
            stats_divs=stats_divs_list
        ),
        code=percentile_js_code + """
        checkbox.active = Array.from({length: n_programs}, (_, i) => i);

        const active_indices = checkbox.active;
        const selected_programs = active_indices.map(i => programs[i]);

        const idx_min = Math.floor(slider_min.value);
        const idx_max = Math.floor(slider_max.value);
        const dates = available_dates.data['dates'];
        const min_date = dates[idx_min];
        const max_date = dates[idx_max];

        const full_data = source_full.data;
        const day_obs = full_data['day_obs'];
        const science_program = full_data['science_program'];
        const filtered_indices = [];

        for (let i = 0; i < day_obs.length; i++) {
            const in_date_range = day_obs[i] >= min_date && day_obs[i] <= max_date;
            const in_program = selected_programs.includes(science_program[i]);

            if (in_date_range && in_program) {
                filtered_indices.push(i);
            }
        }

        const new_data = {};
        for (const key in full_data) {
            new_data[key] = filtered_indices.map(i => full_data[key][i]);
        }
        source.data = new_data;
        source.change.emit();

        // Update statistics
        updateStats(stats_divs, variables, new_data);

        const n_dates_in_range = idx_max - idx_min + 1;
        date_status.text = '<p style="margin: 5px 0; color: #666;">Showing ' + filtered_indices.length + 
                          ' of ' + day_obs.length + ' points | ' + 
                          selected_programs.length + ' of ' + programs.length + ' programs | ' +
                          n_dates_in_range + ' of ' + dates.length + ' dates</p>';

        for (let var_idx = 0; var_idx < variables.length; var_idx++) {
            const hist_source = hist_sources[var_idx];
            if (hist_source) {
                const var_name = variables[var_idx];
                const var_data = new_data[var_name];

                if (!var_data || var_data.length === 0) {
                    hist_source.data = {
                        'top': [],
                        'left': [],
                        'right': [],
                        'bottom': []
                    };
                    hist_source.change.emit();
                    continue;
                }

                const var_min = Math.min(...var_data);
                const var_max = Math.max(...var_data);

                const n_bins = 20;
                const bin_width = (var_max - var_min) / n_bins || 1;
                const bins = new Array(n_bins).fill(0);
                const edges_left = [];
                const edges_right = [];

                for (let i = 0; i < n_bins; i++) {
                    edges_left.push(var_min + i * bin_width);
                    edges_right.push(var_min + (i + 1) * bin_width);
                }

                for (let i = 0; i < var_data.length; i++) {
                    const val = var_data[i];
                    if (!isNaN(val)) {
                        const bin_idx = Math.min(Math.floor((val - var_min) / bin_width), n_bins - 1);
                        if (bin_idx >= 0 && bin_idx < n_bins) {
                            bins[bin_idx]++;
                        }
                    }
                }

                hist_source.data = {
                    'top': bins,
                    'left': edges_left,
                    'right': edges_right,
                    'bottom': new Array(n_bins).fill(0)
                };
                hist_source.change.emit();
            }
        }
    """
    )

    # Deselect all callback
    deselect_all_callback = CustomJS(
        args=dict(
            checkbox=program_checkbox,
            source=source,
            date_status=date_status,
            hist_sources=hist_sources_list,
            variables=variables,
            stats_divs=stats_divs_list
        ),
        code="""
        checkbox.active = [];

        const new_data = {};
        for (const key in source.data) {
            new_data[key] = [];
        }
        source.data = new_data;
        source.change.emit();

        // Clear histograms and stats
        for (let var_idx = 0; var_idx < variables.length; var_idx++) {
            const hist_source = hist_sources[var_idx];
            if (hist_source) {
                hist_source.data = {
                    'top': [],
                    'left': [],
                    'right': [],
                    'bottom': []
                };
                hist_source.change.emit();
            }

            const stats_div = stats_divs[var_idx];
            if (stats_div) {
                stats_div.text = "<span style='font-size:13px; font-family:monospace; color:#999;'>No data</span>";
            }
        }

        date_status.text = '<p style="margin: 5px 0; color: #666;">Showing 0 points | 0 programs selected</p>';
    """
    )

    # Connect callbacks
    date_slider_min.js_on_change('value', slider_callback)
    date_slider_max.js_on_change('value', slider_callback)
    date_text_min.js_on_change('value', text_callback)
    date_text_max.js_on_change('value', text_callback)
    program_checkbox.js_on_change('active', program_filter_callback)

    # Create buttons
    select_all_btn = Button(label="Select All", button_type="success", width=100)
    deselect_all_btn = Button(label="Deselect All", button_type="warning", width=100)
    select_all_btn.js_on_click(select_all_callback)
    deselect_all_btn.js_on_click(deselect_all_callback)

    # Date controls layout
    date_controls = column(
        row(
            column(date_slider_min, date_display_min, date_text_min),
            column(date_slider_max, date_display_max, date_text_max)
        ),
        date_status
    )

    # Variable sliders with stats
    slider_rows = []

    # Add header row
    header_row = row(
        Div(text="<span style='font-size:13px; font-weight:bold;'>Variable Range</span>", width=250, height=25),
        Div(text="<span style='font-size:13px; font-weight:bold;'>Min</span>", width=100, height=25),
        Div(text="<span style='font-size:13px; font-weight:bold;'>Max</span>", width=100, height=25),
        Div(text="<span style='font-size:13px; font-weight:bold;'>5th,  50th,   95th percentile </span>", width=200, height=25)
    )
    slider_rows.append(header_row)

    for idx, (var, label) in enumerate(zip(variables, labels)):
        var_min, var_max = var_bounds[var]

        slider = RangeSlider(
            start=var_min,
            end=var_max,
            value=(var_min, var_max),
            step=(var_max - var_min) / 100,
            title=label,
            width=250
        )

        text_min_var = TextInput(value=f"{var_min:.4f}", title="", width=100)
        text_max_var = TextInput(value=f"{var_max:.4f}", title="", width=100)

        callback_args = {
            'shared_range': shared_ranges[idx],
            'slider': slider,
            'text_min': text_min_var,
            'text_max': text_max_var,
            'source': source,
            'var_name': var,
            'stats_div': stats_divs[idx]
        }

        if idx in hist_sources:
            callback_args['hist_source'] = hist_sources[idx]

            slider_callback_var = CustomJS(args=callback_args, code=percentile_js_code + """
                shared_range.start = slider.value[0];
                shared_range.end = slider.value[1];

                text_min.value = slider.value[0].toFixed(4);
                text_max.value = slider.value[1].toFixed(4);

                const data = source.data;
                const var_data = data[var_name];
                const min_val = slider.value[0];
                const max_val = slider.value[1];

                const filtered = [];
                for (let i = 0; i < var_data.length; i++) {
                    if (var_data[i] >= min_val && var_data[i] <= max_val) {
                        filtered.push(var_data[i]);
                    }
                }

                // Update stats for filtered data in view
                if (filtered.length > 0) {
                    const p5 = computePercentile(filtered, 5);
                    const p50 = computePercentile(filtered, 50);
                    const p95 = computePercentile(filtered, 95);
                    stats_div.text = "<span style='font-size:13px; font-family:monospace;'>" + 
                                    p5.toFixed(3) + "      " + p50.toFixed(3) + "      " + p95.toFixed(3) + "</span>";
                } else {
                    stats_div.text = "<span style='font-size:13px; font-family:monospace; color:#999;'>No data</span>";
                }

                const n_bins = 20;
                const bin_width = (max_val - min_val) / n_bins || 1;
                const bins = new Array(n_bins).fill(0);
                const edges_left = [];
                const edges_right = [];

                for (let i = 0; i < n_bins; i++) {
                    edges_left.push(min_val + i * bin_width);
                    edges_right.push(min_val + (i + 1) * bin_width);
                }

                for (let i = 0; i < filtered.length; i++) {
                    const val = filtered[i];
                    const bin_idx = Math.min(Math.floor((val - min_val) / bin_width), n_bins - 1);
                    if (bin_idx >= 0 && bin_idx < n_bins) {
                        bins[bin_idx]++;
                    }
                }

                hist_source.data['top'] = bins;
                hist_source.data['left'] = edges_left;
                hist_source.data['right'] = edges_right;
                hist_source.data['bottom'] = new Array(n_bins).fill(0);
                hist_source.change.emit();
            """)

            text_callback_var = CustomJS(args=callback_args, code=percentile_js_code + """
                const min_val = parseFloat(text_min.value);
                const max_val = parseFloat(text_max.value);

                if (isNaN(min_val) || isNaN(max_val) || min_val >= max_val) {
                    return;
                }

                slider.value = [min_val, max_val];
                shared_range.start = min_val;
                shared_range.end = max_val;

                const data = source.data;
                const var_data = data[var_name];

                const filtered = [];
                for (let i = 0; i < var_data.length; i++) {
                    if (var_data[i] >= min_val && var_data[i] <= max_val) {
                        filtered.push(var_data[i]);
                    }
                }

                // Update stats for filtered data in view
                if (filtered.length > 0) {
                    const p5 = computePercentile(filtered, 5);
                    const p50 = computePercentile(filtered, 50);
                    const p95 = computePercentile(filtered, 95);
                    stats_div.text = "<span style='font-size:13px; font-family:monospace;'>" + 
                                    p5.toFixed(3) + "      " + p50.toFixed(3) + "      " + p95.toFixed(3) + "</span>";
                } else {
                    stats_div.text = "<span style='font-size:13px; font-family:monospace; color:#999;'>No data</span>";
                }

                const n_bins = 20;
                const bin_width = (max_val - min_val) / n_bins || 1;
                const bins = new Array(n_bins).fill(0);
                const edges_left = [];
                const edges_right = [];

                for (let i = 0; i < n_bins; i++) {
                    edges_left.push(min_val + i * bin_width);
                    edges_right.push(min_val + (i + 1) * bin_width);
                }

                for (let i = 0; i < filtered.length; i++) {
                    const val = filtered[i];
                    const bin_idx = Math.min(Math.floor((val - min_val) / bin_width), n_bins - 1);
                    if (bin_idx >= 0 && bin_idx < n_bins) {
                        bins[bin_idx]++;
                    }
                }

                hist_source.data['top'] = bins;
                hist_source.data['left'] = edges_left;
                hist_source.data['right'] = edges_right;
                hist_source.data['bottom'] = new Array(n_bins).fill(0);
                hist_source.change.emit();
            """)
        else:
            slider_callback_var = CustomJS(args=callback_args, code=percentile_js_code + """
                shared_range.start = slider.value[0];
                shared_range.end = slider.value[1];
                text_min.value = slider.value[0].toFixed(4);
                text_max.value = slider.value[1].toFixed(4);

                const data = source.data;
                const var_data = data[var_name];
                const min_val = slider.value[0];
                const max_val = slider.value[1];

                const filtered = var_data.filter(v => v >= min_val && v <= max_val);

                if (filtered.length > 0) {
                    const p5 = computePercentile(filtered, 5);
                    const p50 = computePercentile(filtered, 50);
                    const p95 = computePercentile(filtered, 95);
                    stats_div.text = "<span style='font-size:13px; font-family:monospace;'>" + 
                                    p5.toFixed(3) + "      " + p50.toFixed(3) + "      " + p95.toFixed(3) + "</span>";
                } else {
                    stats_div.text = "<span style='font-size:13px; font-family:monospace; color:#999;'>No data</span>";
                }
            """)

            text_callback_var = CustomJS(args=callback_args, code=percentile_js_code + """
                const min_val = parseFloat(text_min.value);
                const max_val = parseFloat(text_max.value);

                if (isNaN(min_val) || isNaN(max_val) || min_val >= max_val) {
                    return;
                }

                slider.value = [min_val, max_val];
                shared_range.start = min_val;
                shared_range.end = max_val;

                const data = source.data;
                const var_data = data[var_name];

                const filtered = var_data.filter(v => v >= min_val && v <= max_val);

                if (filtered.length > 0) {
                    const p5 = computePercentile(filtered, 5);
                    const p50 = computePercentile(filtered, 50);
                    const p95 = computePercentile(filtered, 95);
                    stats_div.text = "<span style='font-size:13px; font-family:monospace;'>" + 
                                    p5.toFixed(3) + "      " + p50.toFixed(3) + "      " + p95.toFixed(3) + "</span>";
                } else {
                    stats_div.text = "<span style='font-size:13px; font-family:monospace; color:#999;'>No data</span>";
                }
            """)

        slider.js_on_change('value', slider_callback_var)
        text_min_var.js_on_change('value', text_callback_var)
        text_max_var.js_on_change('value', text_callback_var)

        # Add stats div to the row
        control_row = row(slider, text_min_var, text_max_var, stats_divs[idx])
        slider_rows.append(control_row)

    sliders_panel = column(*slider_rows)

    # Title
    unique_bands = sorted(set(bands))
    legend_html = ' '.join([f'<span style="color:{band_colors[band]};">●</span> {band}' 
                            for band in unique_bands])

    title_div = Div(text=f"""
    <div style="text-align: center; font-family: Arial, sans-serif;">
        <h2 style="margin-bottom: 5px;">Observing Blocks Parameter Space</h2>
        <p style="margin-top: 5px; margin-bottom: 10px;">Available dates: {available_dates[0]} - {available_dates[-1]} ({n_dates} dates)</p>
        <p style="margin-top: 5px;"><strong>Band:</strong> {legend_html}</p>
    </div>
    """, width=n_vars * plot_width, height=100)

    # Program selection panel
    program_label = Div(text="<strong>Select Science Programs:</strong>", width=200, height=20)
    program_controls = column(
        program_label,
        row(select_all_btn, deselect_all_btn),
        program_checkbox
    )

    # Separators
    separator1 = Div(text="<hr style='margin: 20px 0; border: none; border-top: 2px solid #667eea;'>", 
                     width=n_vars * plot_width)
    separator2 = Div(text="<hr style='margin: 20px 0; border: none; border-top: 1px solid #dee2e6;'>", 
                     width=n_vars * plot_width)
    separator3 = Div(text="<hr style='margin: 20px 0; border: none; border-top: 1px solid #dee2e6;'>", 
                     width=n_vars * plot_width)

    grid = gridplot(plots, toolbar_location='right')

    # Final layout
    layout = column(
        title_div, 
        separator1,
        program_controls,
        separator2,
        date_controls,
        separator3,
        sliders_panel, 
        grid
    )

    show(layout)

    print(f"Available dates: {n_dates}")
    print(f"Date range: {available_dates[0]} - {available_dates[-1]}")
    print(f"Bands: {', '.join(unique_bands)}")
    print(f"Science programs: {len(unique_programs)}")